# 08 — Random Forest + 4 Katmanlı Validation (Hafta 10)

**Hedef:** LST = f(7 bağımsız değişken + saturated flag) Random Forest regresyonu.

**Validation katmanları:**
1. **Random 5-fold CV** — klasik, hücre seviyesinde. Yüksek skor beklenir (mekânsal autocorrelation).
2. **Spatial Block CV (500 m blocks)** — bloklar fold'a atanır → autocorrelation control. Random'dan düşük olur.
3. **Mahalle Hold-out** — 3 mahalle (ALTINKUM, HURMA, GÜRSU) tamamen test'e ayrılır → out-of-area generalization.
4. **Permutation Null Test** — target shuffled, 50 perm; gerçek R² null 99p'den yüksek olmalı (p<0.01).

5. **(Hafta 11) Yıllar arası CV** — yıl bazlı LST kompozit gerekir, bu notebook'ta yok.

**Önkoşul:** `grid_30m_modeling.gpkg` (Hafta 9).

**Çıktılar:**
- `data/processed/grid_30m_predictions.gpkg`
- `results/rf_model.pkl`
- `results/rf_validation.json`
- `figures/08_*.png`
- `tables/08_rf_metrics.csv`

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.config import (
    DATA_PROCESSED, RESULTS, FIGURES, TABLES,
    GRID_30M_MODELING, GRID_30M_BUILDING,
    SELECTED_FEATURES, TARGET_COLUMN,
    RF_N_ESTIMATORS, RANDOM_STATE,
    SPATIAL_CV_BLOCK_M, HOLDOUT_NEIGHBORHOODS, CRS_PROJECTED,
    PREDICTIONS_GPKG, MODEL_PKL, RF_VALIDATION_JSON,
)
from src.modeling import (
    prepare_modeling_matrix, train_rf,
    random_kfold_cv, spatial_block_cv,
    neighborhood_holdout, permutation_null_test,
    feature_importance,
)

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- Yükle + mahalle bilgisi ekle ---
gdf = gpd.read_file(GRID_30M_MODELING)
print(f"Grid: {len(gdf):,} hücre, {len(gdf.columns)} kolon")

# Mahalle bilgisini imar_pilot'tan en yakın nokta ile bağla
imar = gpd.read_file(DATA_PROCESSED / "imar_pilot.gpkg")
joined = gpd.sjoin_nearest(gdf[["cell_id", "geometry"]], imar[["mahalle", "geometry"]],
                            how="left", max_distance=200)
mahalle_per_cell = joined.groupby("cell_id")["mahalle"].first()
gdf["mahalle"] = gdf["cell_id"].map(mahalle_per_cell)
print(f"Mahalle ataması: {gdf['mahalle'].notna().sum():,} / {len(gdf):,}")
print(gdf['mahalle'].value_counts().head(10))

In [ ]:
# --- Modelleme matrisi ---
X, y, feat_names = prepare_modeling_matrix(
    gdf, features=SELECTED_FEATURES, target=TARGET_COLUMN,
    fillna_zero=True, add_saturated_flag=True,
)
print(f"X: {X.shape}  y: {y.shape}")
print(f"Features ({len(feat_names)}): {feat_names}")
print(f"is_dtc_saturated count: {int(X['is_dtc_saturated'].sum()):,}")
print(f"X NaN total: {X.isna().sum().sum()} (0 olmalı)")
print(f"\nTarget özet:")
print(y.describe().round(2))

In [ ]:
# --- Validation Layer 1: Random 5-fold CV ---
print(f"5-fold random CV (n_estimators={RF_N_ESTIMATORS})...")
random_cv = random_kfold_cv(X, y, n_splits=5,
                             random_state=RANDOM_STATE,
                             n_estimators=RF_N_ESTIMATORS)
print(f"\nMean: RMSE={random_cv['mean']['rmse']:.3f}  "
      f"MAE={random_cv['mean']['mae']:.3f}  R²={random_cv['mean']['r2']:.3f}")
print(f"Std:  RMSE={random_cv['std']['rmse']:.3f}  "
      f"MAE={random_cv['std']['mae']:.3f}  R²={random_cv['std']['r2']:.3f}")

In [ ]:
# --- Validation Layer 2: Spatial Block CV ---
print(f"Spatial Block CV ({SPATIAL_CV_BLOCK_M}m bloklar)...")
spatial_cv = spatial_block_cv(X, y, gdf.geometry,
                               block_size_m=SPATIAL_CV_BLOCK_M,
                               n_splits=5,
                               random_state=RANDOM_STATE,
                               n_estimators=RF_N_ESTIMATORS)
print(f"\nN unique blocks: {spatial_cv['n_blocks']:,}")
print(f"Mean: RMSE={spatial_cv['mean']['rmse']:.3f}  "
      f"MAE={spatial_cv['mean']['mae']:.3f}  R²={spatial_cv['mean']['r2']:.3f}")

gap = random_cv['mean']['r2'] - spatial_cv['mean']['r2']
print(f"\nR² gap (random - spatial): {gap:+.3f}")
if gap > 0.10:
    print("⚠ Mekânsal autocorrelation overfitting riski (gap > 0.10)")

In [ ]:
# --- Validation Layer 3: Mahalle Hold-out ---
print(f"Mahalle hold-out: {HOLDOUT_NEIGHBORHOODS}")
holdout = neighborhood_holdout(X, y, gdf['mahalle'],
                                test_neighborhoods=HOLDOUT_NEIGHBORHOODS,
                                n_estimators=RF_N_ESTIMATORS,
                                random_state=RANDOM_STATE)
print(f"\nTrain: {holdout['n_train']:,}  Test: {holdout['n_test']:,}")
print(f"Test metrics: RMSE={holdout['metrics']['rmse']:.3f}  "
      f"MAE={holdout['metrics']['mae']:.3f}  R²={holdout['metrics']['r2']:.3f}")

In [ ]:
# --- Validation Layer 4: Permutation Null Test ---
print("Permutation null test (50 perm × 3-fold CV; ~3-5 dk sürebilir)...")
null_test = permutation_null_test(X, y, n_permutations=50,
                                    n_estimators=200,
                                    random_state=RANDOM_STATE)
print(f"\nNull R² mean: {null_test['null_r2_mean']:.3f}  "
      f"std: {null_test['null_r2_std']:.3f}")
print(f"Null R² 99th percentile: {null_test['null_r2_99p']:.3f}")
print(f"Real R² (random CV): {random_cv['mean']['r2']:.3f}")

significant = random_cv['mean']['r2'] > null_test['null_r2_99p']
print(f"\nGerçek > Null 99p: {significant} → "
      + ("p < 0.01 ✓ anlamlı" if significant else "⚠ anlamsız"))

In [ ]:
# --- Final model + feature importance ---
print("Final RF tüm veride eğitiliyor...")
final_model = train_rf(X, y, n_estimators=RF_N_ESTIMATORS,
                        random_state=RANDOM_STATE)

print("Permutation importance (10 repeat)...")
fi = feature_importance(final_model, X, y, n_repeats=10,
                         random_state=RANDOM_STATE)
print("\nFeature importance (permutation sıralı):")
print(fi.to_string(index=False))

joblib.dump(final_model, MODEL_PKL)
print(f"\nKaydedildi: {MODEL_PKL.name}")

In [ ]:
# --- Tahmin + residual + GPKG ---
gdf['predicted_lst'] = final_model.predict(X)
gdf['residual'] = gdf[TARGET_COLUMN] - gdf['predicted_lst']

out = gdf[['cell_id', TARGET_COLUMN, 'predicted_lst', 'residual',
           'mahalle', 'geometry']]
out.to_file(PREDICTIONS_GPKG, driver="GPKG")
print(f"Kaydedildi: {PREDICTIONS_GPKG.name} ({PREDICTIONS_GPKG.stat().st_size/1024/1024:.2f} MB)")

# Saturated hücreler RMSE
sat_mask = X['is_dtc_saturated'] == 1
sat_rmse = float(np.sqrt(((gdf[TARGET_COLUMN][sat_mask] - gdf['predicted_lst'][sat_mask])**2).mean()))
nonsat_rmse = float(np.sqrt(((gdf[TARGET_COLUMN][~sat_mask] - gdf['predicted_lst'][~sat_mask])**2).mean()))
print(f"\nSaturated hücreler  ({sat_mask.sum():,}): RMSE = {sat_rmse:.3f}")
print(f"Non-saturated     ({(~sat_mask).sum():,}): RMSE = {nonsat_rmse:.3f}")
print(f"Fark: {sat_rmse - nonsat_rmse:+.3f} m  (sınırlılık raporu)")

In [ ]:
# --- Görselleştirme: 4 panel ---
fig, axes = plt.subplots(2, 2, figsize=(15, 13))

# 1. CV bar
ax = axes[0, 0]
metrics_data = pd.DataFrame([
    {'katman': 'Random CV', 'R²': random_cv['mean']['r2'], 'RMSE': random_cv['mean']['rmse']},
    {'katman': 'Spatial CV', 'R²': spatial_cv['mean']['r2'], 'RMSE': spatial_cv['mean']['rmse']},
    {'katman': 'Hold-out', 'R²': holdout['metrics']['r2'], 'RMSE': holdout['metrics']['rmse']},
    {'katman': 'Null 99p', 'R²': null_test['null_r2_99p'], 'RMSE': np.nan},
])
x = np.arange(len(metrics_data))
ax.bar(x, metrics_data['R²'], color=['#2A9D8F', '#E76F51', '#264653', '#aaaaaa'])
ax.set_xticks(x)
ax.set_xticklabels(metrics_data['katman'], fontsize=9)
ax.set_ylabel('R²')
ax.set_title('Validation R² karşılaştırma')
for i, v in enumerate(metrics_data['R²']):
    if not np.isnan(v):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=8)

# 2. Feature importance
ax = axes[0, 1]
fi_sorted = fi.sort_values('permutation_importance_mean')
y_pos = np.arange(len(fi_sorted))
ax.barh(y_pos, fi_sorted['permutation_importance_mean'],
        xerr=fi_sorted['permutation_importance_std'],
        color='#2A9D8F', edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(fi_sorted['feature'], fontsize=9)
ax.set_xlabel('Permutation importance (R² düşüşü)')
ax.set_title('Feature importance')

# 3. Predicted vs actual choropleth
ax = axes[1, 0]
vmin = min(gdf[TARGET_COLUMN].min(), gdf['predicted_lst'].min())
vmax = max(gdf[TARGET_COLUMN].max(), gdf['predicted_lst'].max())
gdf.plot(column='predicted_lst', cmap='inferno', ax=ax, linewidth=0,
         legend=True, legend_kwds={'label': 'Predicted LST (°C)', 'shrink': 0.6},
         vmin=vmin, vmax=vmax)
ax.set_title('RF Predicted LST haritası')
ax.set_aspect('equal')
ax.ticklabel_format(style='plain')

# 4. Residual choropleth
ax = axes[1, 1]
rmax = max(abs(gdf['residual'].quantile(0.02)), abs(gdf['residual'].quantile(0.98)))
gdf.plot(column='residual', cmap='RdBu_r', ax=ax, linewidth=0,
         legend=True, legend_kwds={'label': 'Residual (actual - pred)', 'shrink': 0.6},
         vmin=-rmax, vmax=rmax)
ax.set_title('Residual haritası')
ax.set_aspect('equal')
ax.ticklabel_format(style='plain')

plt.tight_layout()
plt.savefig(FIGURES / '08_rf_overview.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# --- Validation JSON + Metrics CSV ---
validation_results = {
    'random_cv': {'mean': random_cv['mean'], 'std': random_cv['std']},
    'spatial_cv': {'mean': spatial_cv['mean'], 'std': spatial_cv['std'],
                   'n_blocks': spatial_cv['n_blocks'],
                   'block_size_m': spatial_cv['block_size_m']},
    'holdout': holdout,
    'permutation_null': {k: v for k, v in null_test.items() if k != 'samples'},
    'saturated_rmse': sat_rmse,
    'nonsat_rmse': nonsat_rmse,
    'feature_importance': fi.to_dict(orient='records'),
    'n_features': int(X.shape[1]),
    'n_samples': int(X.shape[0]),
    'rf_n_estimators': RF_N_ESTIMATORS,
    'random_state': RANDOM_STATE,
}
with open(RF_VALIDATION_JSON, 'w', encoding='utf-8') as f:
    json.dump(validation_results, f, indent=2, ensure_ascii=False)
print(f'Saved: {RF_VALIDATION_JSON.name}')

metrics_csv = pd.DataFrame([
    {'katman': 'Random 5-fold CV', 'RMSE': random_cv['mean']['rmse'],
     'MAE': random_cv['mean']['mae'], 'R2': random_cv['mean']['r2']},
    {'katman': f'Spatial Block CV ({SPATIAL_CV_BLOCK_M}m)',
     'RMSE': spatial_cv['mean']['rmse'],
     'MAE': spatial_cv['mean']['mae'],
     'R2': spatial_cv['mean']['r2']},
    {'katman': f'Hold-out {HOLDOUT_NEIGHBORHOODS}',
     'RMSE': holdout['metrics']['rmse'],
     'MAE': holdout['metrics']['mae'],
     'R2': holdout['metrics']['r2']},
    {'katman': 'Null R² 99p', 'RMSE': np.nan, 'MAE': np.nan,
     'R2': null_test['null_r2_99p']},
]).round(3)
metrics_csv.to_csv(TABLES / '08_rf_metrics.csv', index=False, encoding='utf-8-sig')
print(f'Saved: 08_rf_metrics.csv')
metrics_csv

## Özet

- 8 özellikli RF (7 raw değişken + `is_dtc_saturated` flag) eğitildi.
- 4 validation katmanı: random CV, spatial blok CV, mahalle hold-out, permutation null.
- Final model `results/rf_model.pkl` olarak kaydedildi.
- 30 m grid'de tahmin ve residual hesaplandı, `grid_30m_predictions.gpkg` olarak yazıldı.

## Sonraki Adım

- **Hafta 11** — Yıllar arası CV (LST'yi yıl bazlı yeniden çek + 5 ayrı model).
- **Hafta 12** — `09_shap_analysis.ipynb` (SHAP değerleri ile interpretability) ve `10_utpm_index.ipynb` (UTPM lineer indeks).